In [1]:
import os
import time
import signal
import gymnasium
import subprocess
import numpy as np
from input_controller import InputController 
from udp_event_buffer import UdpEventBuffer

In [2]:
class HKRLEnv(gymnasium.Env):
    def __init__(self,
        display  = ":577",
        udp_host = "127.0.0.1",
        udp_port = 28115,
        frame_size = (256, 256),
        launch_cmd_xvfb    = None,
        launch_cmd_proton  = None,
        launch_cwd         = None,
        launch_data_path   = None,
        launch_env         = None,
        option_frame       = 9
    ):

        # >>> init 
        self.display  = display
        self.udp_host = udp_host
        self.udp_port = int(udp_port)
        self.frame_w, self.frame_h = int(frame_size[0]), int(frame_size[1])
        self.launch_cmd_xvfb   = launch_cmd_xvfb
        self.launch_cmd_proton = launch_cmd_proton
        self.launch_cwd        = launch_cwd
        self.launch_data_path = launch_data_path
        self.launch_env       = launch_env
        self.last_step_time = time.time()
        self.option_frame   = option_frame

        # Internal state
        self._alive = False
        self._is_first_time = True
        # <<<

        # >>> action space
        # self.keys_count: int = int(os.getenv("HKRL_KEYS_COUNT", "79"))
        # self.action_space = gymnasium.spaces.Dict({
        #     "key_list":     gymnasium.spaces.MultiBinary(self.keys_count),
        #     "mouse_list":   gymnasium.spaces.Box(low=0.0, high=1.0, shape=(5,), dtype=np.float32),
        # })
        self.action_space = gymnasium.spaces.Discrete(2**9)
        # <<<

        # >>> observation space
        # Observation: stacked frames, channel-last (H, W, 3*stack). We will resize to (frame_h, frame_w).
        self.observation_space = gymnasium.spaces.Box(
            low=0,
            high=255,
            shape=(self.frame_h, self.frame_w, 3),
            dtype=np.uint8,
        )
        # <<<

    # 启动停止 截屏\键鼠操作的控制器
    def _start_controller(self, display):
        self.inputcontroller = InputController(display)
        self.inputcontroller.start()
        self._action_empty()

    def _stop_controller(self):
        try:
            self.inputcontroller.stop()
            self.inputcontroller = None
        except:
            pass
        
    # 启动停止 xvfb与proton
    def _start_xvfb_proton(self, display, launch_cmd_xvfb, launch_cmd_proton, launch_cwd, launch_env):
        self.xvfb_proc = subprocess.Popen(
            launch_cmd_xvfb + [display],
            cwd=launch_cwd,
            env=launch_env,
            preexec_fn=os.setsid,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        self.proton_proc = subprocess.Popen(
            launch_cmd_proton,
            cwd=launch_cwd,
            env=launch_env,
            preexec_fn=os.setsid,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        
    def _stop_xvfb_proton(self):
        try:
            # 如果proton进程活着
            while self.proton_proc.poll() == None:
                # 杀死proton
                self.launch_env["WINEPREFIX"] = os.path.join(self.launch_env["STEAM_COMPAT_DATA_PATH"], "pfx")
                subprocess.run(
                    ["/opt/GE-Proton10-27/files/bin/wineserver", "-k"],
                    env=self.launch_env,
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL,
                )
            self.proton_proc = None
            # 如果xvfb进程活着
            while self.xvfb_proc.poll() == None:
                # 杀死xvfb
                os.killpg(self.xvfb_proc.pid, signal.SIGTERM)
            self.xvfb_proc = None
        except:
            pass

    # 启动停止 udp监听
    def _start_udp_listener(self, host, port):
        self.udp_listener = UdpEventBuffer(host, port)
        self.udp_listener.start()
        
    def _stop_udp_linstener(self):
        try:
            self.udp_listener.stop()
            self.udp_listener = None
        except:
            pass

    def _maybe_resize(self, img, target_wh):
        """进行重采样"""
        h, w, c = img.shape
        tw, th = target_wh
        if (w, h) == (tw, th):
            return img
        # Nearest-neighbor resize without extra deps
        x_idx = (np.linspace(0, w - 1, tw)).astype(np.int64)
        y_idx = (np.linspace(0, h - 1, th)).astype(np.int64)
        return img[y_idx][:, x_idx]

    # 将数字转化为list
    @staticmethod
    def _number2list(action):
        action = int(action)
        return_list = []
        for i in range(9):
            return_list += [action % 2]
            action //= 2
        return return_list

    # 隔固定时间开始运行
    def _step_time_wait(self):
        period = 1.0 / self.option_frame
        now = time.perf_counter()
        target = self.last_step_time + period
        
        if now < target:
            # 粗睡, 留 1ms 给忙等
            time.sleep(max(0.0, target - now - 0.001))
            while time.perf_counter() < target:
                pass
            self.last_step_time = target
        else:
            # 跑慢了就直接追上当前时间
            self.last_step_time = now
            
    # Gym的固定API
    def reset(self, *, seed=None, options=None):
        """重启环境"""
        super().reset(seed=seed)
        # 如果是第一次启动游戏, 则启动xvfb+proton
        if self._is_first_time:
            self._is_first_time = False
            self._start_udp_listener(self.udp_host, self.udp_port)
            self._start_xvfb_proton(self.display, self.launch_cmd_xvfb, self.launch_cmd_proton, self.launch_cwd, self.launch_env)
            self._start_controller(self.display)
            time.sleep(10)
            # 选择存档
            self._menu_choose()
            time.sleep(10)
            # 此处为走到三螳螂雕像下
            self._walk_to_Mantis_Lords()
        else:
            time.sleep(10)
        # 进入游戏
        self._fight_again(1)
        
        # 等待接收到场景变化, 且变为GG_Workshop就代表开始
        break_sign = False
        while not break_sign:
            message_list = self.udp_listener.drain_events()
            
            for message_dict in message_list:
                if message_dict['type'] == 'scene_changed' and message_dict['scene'] == 'GG_Workshop':
                    break_sign = True
                    break
        
        self.enemy_dict = {}
        for message_dict in message_list:
            if message_dict["type"] == "scene_changed": 
                for enemy in message_dict["snapshot"]["enemies"]:
                    if self.enemy_dict.get(enemy['id']) == None:
                        self.enemy_dict[enemy['id']] = {
                            "name":  enemy['name'],
                            "hp":    enemy['hp'],
                            "hpMax": enemy['hp']
                        }
                        
        frame = self.inputcontroller.get_frame()
        # frame = self._maybe_resize(frame, (self.frame_w, self.frame_h))
        info = {}
        self._alive = True
        self._reward = 0
        self.last_hp = 9
        self.last_step_time = time.perf_counter()
        return frame, info

    def step(self, action):
        # 传入动作为0到511的数字, 将其转化为长为9的list
        action = self._number2list(action)
        
        # # 传入动作list, 长度为9, 确保按键均为0或1
        # for idx, act in enumerate(action):
        #     if act > 0.5:
        #         action[idx] = 1
        #     else:
        #         action[idx] = 0

        # 留下的键位为 w a s d f j k l space
        action_dict = {
            "key_list":   [0 for _ in range(29)] + [action[0]] + \
                          [0 for _ in range(12)] + action[1:5] + \
                          [0 for _ in range( 2)] + action[5:8] + [0 for _ in range(16)] + [action[8]] + [0 for _ in range(11)],
            "mouse_list": [0 for _ in range(5)]
        }

        # 发送到控制器
        self.inputcontroller.send_action(action_dict)

        # 接受 udp_listener 的返回报文
        events_list = self.udp_listener.drain_events()
        terminated = False
        info_delta = {}
        old_reward = self._reward
        for events in events_list:
            terminated, info_delta = self._compute_reward_from_events(events)

        # Capture new frame and update stack
        frame = self.inputcontroller.get_frame()
        # frame = self._maybe_resize(frame, (self.frame_w, self.frame_h))

        self._step_time_wait()

        return frame, self._reward - old_reward, terminated, False, info_delta
    
    def close(self):
        """关闭环境"""
        self._stop_controller()
        self._stop_xvfb_proton()
        self._stop_udp_linstener()

    def _compute_reward_from_events(self, events):
        # 如果udp报文为空, 则奖励不变
        if events == []:
            return False, None

        terminated = False
        info = {}
        message_dict = events
        # 如果udp报文类型为场景变化, 则结束游戏
        if events['type'] == 'scene_changed':
            self._alive = False
            terminated = True
            message_dict = events["snapshot"]
            self._action_empty()
        else:
            # 若不为空则计算奖励
            ## 首先计算角色血量变化奖励, 扣血减去 扣血量*10, 加血加上 加血量*10
            ### 如果当前血量大于之前血量
            if message_dict["player"]["hp"] > self.last_hp:
                # 如果是回了一滴血造成的
                if message_dict["player"]["hp"] - self.last_hp == 1:
                    self._reward += 10
                # 如果回了两滴及以上说明是死亡后自动回满血, 不计算奖励
            ### 如果当前血量小于等于之前血量, 计算损失(可能2血boss或辐辉难度)
            else:
                self._reward += (message_dict["player"]["hp"] - self.last_hp) * 10
            self.last_hp = message_dict["player"]["hp"]
            ## 然后计算攻击敌人奖励, 每打掉1点血+0.1奖励
            for enemy in message_dict["enemies"]:
                # 如果这个敌人之前没有记录, 则加入存储中
                if self.enemy_dict.get(enemy['id']) == None:
                    self.enemy_dict[enemy['id']] = {
                        "name":  enemy['name'],
                        "hp":    enemy['hp'],
                        "hpMax": enemy['hp']
                    }
                # 如果这个敌人之前存在记录, 则计算奖励
                else:
                    # 如果敌人的hp出现上升, 代表之前没有捕捉到敌人的血量初始化
                    if self.enemy_dict[enemy['id']]['hp'] < enemy['hp']:
                        # 血量发生变化说明之前攻击过敌人, 默认用骨钉打的
                        self._reward += 2.1
                        self.enemy_dict[enemy['id']]['hp'] = enemy['hp']
                        self.enemy_dict[enemy['id']]['hpMax'] = enemy['hp'] + 21
                    # 如果敌人的hp不变或下降, 则计算奖励
                    else:
                        self._reward += (self.enemy_dict[enemy['id']]['hp'] - enemy['hp']) * 0.1
                        self.enemy_dict[enemy['id']]['hp'] = enemy['hp']
        
        return terminated, info
    
    def _menu_choose(self):
        """菜单界面选择第二个存档"""
        time.sleep(30)
        # 按下空格0.2秒
        send_action = {
            "key_list":   [0 for _ in range(29)] + [0] + \
                          [0 for _ in range(12)] + [0, 0, 0, 0] + \
                          [0 for _ in range( 2)] + [0, 0, 0] + \
                          [0 for _ in range(16)] + [1] + [0 for _ in range(11)],
            "mouse_list": [0 for _ in range(5)]
        }
        self.inputcontroller.send_action(send_action)
        time.sleep(0.2)
        self._action_empty()
        time.sleep(2)
        # 按下向下0.2秒
        send_action = {
            "key_list":   [0 for _ in range(29)] + [0] + \
                          [0 for _ in range(12)] + [0, 1, 0, 0] + \
                          [0 for _ in range( 2)] + [0, 0, 0] + \
                          [0 for _ in range(16)] + [0] + [0 for _ in range(11)],
            "mouse_list": [0 for _ in range(5)]
        }
        self.inputcontroller.send_action(send_action)
        time.sleep(0.2)
        self._action_empty()
        time.sleep(0.2)
        # 按下空格0.2秒
        send_action = {
            "key_list":   [0 for _ in range(29)] + [0] + \
                          [0 for _ in range(12)] + [0, 0, 0, 0] + \
                          [0 for _ in range( 2)] + [0, 0, 0] + \
                          [0 for _ in range(16)] + [1] + [0 for _ in range(11)],
            "mouse_list": [0 for _ in range(5)]
        }
        self.inputcontroller.send_action(send_action)
        time.sleep(0.2)
        self._action_empty()
        
    def _walk_to_Mantis_Lords(self):
        """走到三螳螂雕像位置"""
        # 按下跳跃2秒, 从椅子上起身
        send_action = {
            "key_list":   [0 for _ in range(29)] + [0] + \
                          [0 for _ in range(12)] + [0, 0, 0, 0] + \
                          [0 for _ in range( 2)] + [0, 0, 0] + \
                          [0 for _ in range(16)] + [1] + [0 for _ in range(11)],
            "mouse_list": [0 for _ in range(5)]
        }
        self.inputcontroller.send_action(send_action)
        time.sleep(2)
        self._action_empty()
        
        # 按下向右2秒, 跳下平台
        send_action = {
            "key_list":   [0 for _ in range(29)] + [0] + \
                          [0 for _ in range(12)] + [0, 0, 1, 0] + \
                          [0 for _ in range( 2)] + [0, 0, 0] + \
                          [0 for _ in range(16)] + [0] + [0 for _ in range(11)],
            "mouse_list": [0 for _ in range(5)]
        }
        self.inputcontroller.send_action(send_action)
        time.sleep(2)
        self._action_empty()
        
        # 坠落时间
        time.sleep(1.5)
        
        # 按下向右8.5秒, 走到三螳螂雕像位置
        send_action = {
            "key_list":   [0 for _ in range(29)] + [0] + \
                          [0 for _ in range(12)] + [0, 0, 1, 0] + \
                          [0 for _ in range( 2)] + [0, 0, 0] + \
                          [0 for _ in range(16)] + [0] + [0 for _ in range(11)],
            "mouse_list": [0 for _ in range(5)]
        }
        self.inputcontroller.send_action(send_action)
        time.sleep(8.5)
        self._action_empty()
    
    def _fight_again(self, level):
        # 按下空格0.2秒
        send_action = {
            "key_list":   [0 for _ in range(29)] + [0] + \
                          [0 for _ in range(12)] + [0, 0, 0, 0] + \
                          [0 for _ in range( 2)] + [0, 0, 0] + \
                          [0 for _ in range(16)] + [1] + [0 for _ in range(11)],
            "mouse_list": [0 for _ in range(5)]
        }
        self.inputcontroller.send_action(send_action)
        time.sleep(0.2)
        self._action_empty()
        time.sleep(3)
        for _ in range(level):
            # 按下向下0.1秒
            send_action = {
                "key_list":   [0 for _ in range(29)] + [0] + \
                              [0 for _ in range(12)] + [0, 1, 0, 0] + \
                              [0 for _ in range( 2)] + [0, 0, 0] + \
                              [0 for _ in range(16)] + [0] + [0 for _ in range(11)],
                "mouse_list": [0 for _ in range(5)]
            }
            self.inputcontroller.send_action(send_action)
            time.sleep(0.1)
            self._action_empty()
            time.sleep(1)
        # 按下空格0.2秒
        send_action = {
            "key_list":   [0 for _ in range(29)] + [0] + \
                          [0 for _ in range(12)] + [0, 0, 0, 0] + \
                          [0 for _ in range( 2)] + [0, 0, 0] + \
                          [0 for _ in range(16)] + [1] + [0 for _ in range(11)],
            "mouse_list": [0 for _ in range(5)]
        }
        self.inputcontroller.send_action(send_action)
        time.sleep(0.2)
        self._action_empty()
        
    def _action_empty(self):
        """释放所有按键"""
        send_action = {
            "key_list":   [0 for _ in range(29)] + [0] + \
                          [0 for _ in range(12)] + [0, 0, 0, 0] + \
                          [0 for _ in range( 2)] + [0, 0, 0] + \
                          [0 for _ in range(16)] + [0] + [0 for _ in range(11)],
            "mouse_list": [0 for _ in range(5)]
        }
        self.inputcontroller.send_action(send_action)

In [3]:
def make_hkrl_env(
    display  = ":577",      # 预备打开的窗口
    udp_host = "127.0.0.1", # 发送 udp 报文(游戏mod发送)的地址
    udp_port = 28115,       # 监听 udp 端口
    frame_size = (256, 256),  # 裁剪尺寸大小
    # xvfb与proton的启动命令
    launch_cmd_xvfb   = ["Xvfb", "-screen", "0", "1280x960x24", "-ac", "-nolisten", "tcp"],
    launch_cmd_proton = ["/opt/GE-Proton10-27/proton", "waitforexitandrun", "/workspace/compatdata/Hollow.Knight.GOG/pfx/drive_c/GOG Games/Hollow Knight/Hollow Knight.exe"],
    launch_cwd        = "/workspace",                             # 线程目录
    launch_data_path  = "/workspace/compatdata/Hollow.Knight.GOG", # Proton工作的目录
    launch_env        = None,                                     # 游戏启动环境变量
):
    return HKRLEnv(
        display=display,
        udp_host=udp_host,
        udp_port=udp_port,
        frame_size=frame_size,
        launch_cmd_xvfb=launch_cmd_xvfb,
        launch_cmd_proton=launch_cmd_proton,
        launch_cwd=launch_cwd,
        launch_data_path=launch_data_path,
        launch_env=launch_env,
    )

In [4]:
os_env = os.environ.copy()
os_env["STEAM_COMPAT_DATA_PATH"] = "/workspace/compatdata/Hollow.Knight.GOG"
hkrl_env = make_hkrl_env(launch_env=os_env)

In [10]:
obs, info = hkrl_env.reset()

In [11]:
hkrl_env.step(0)

(array([[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        ...,
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]]], dtype=uint8),
 -17.9,
 False,
 False,
 {})

In [12]:
hkrl_env.close()